Validador do modelo quantizado TFLite

In [10]:
import tensorflow as tf
import numpy as np
import pickle
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
with open('Dados_EEG\dados_EEG1.pkl', mode='rb') as f:
  X_train, y_train, X_test, y_test, CANAIS, TAM_JANELA= pickle.load(f)


#  modelo .tflite 
model_path = "modelos\model_mov-imag.tflite"
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

if len(y_test.shape) == 2:
    true_labels = np.argmax(y_test, axis=1)
else:
    true_labels = y_test

In [ ]:
input_type = input_details[0]['dtype']
print(f"Tipo de entrada esperado pelo modelo: {input_type}") # Verifique se modelo é quantizado


input_scale, input_zero_point = input_details[0]['quantization']

X_test_quantized = X_test 

if (input_scale, input_zero_point) != (0.0, 0): 
    print("Modelo é quantizado. Aplicando quantização ao X_test...")
    
    X_test_quantized = (X_test / input_scale) + input_zero_point  
    X_test_quantized = X_test_quantized.astype(input_type)
else:
    print("Modelo não é quantizado (float32).")

Tipo de entrada esperado pelo modelo: <class 'numpy.int8'>
Modelo é quantizado. Aplicando quantização ao X_test...


In [ ]:
predictions_tflite = []
total_samples = X_test_quantized.shape[0]

print(f"Iniciando inferência em {total_samples} amostras...")

for i in range(total_samples):

    sample = X_test_quantized[i]
    sample_batch = np.expand_dims(sample, axis=0).astype(input_type)  #dimensão batch  
    interpreter.set_tensor(input_details[0]['index'], sample_batch)
    interpreter.invoke() #inferência
    output_data = interpreter.get_tensor(output_details[0]['index'])
  
    predicted_class = np.argmax(output_data[0])
    predictions_tflite.append(predicted_class)

print("Inferência concluída.")

Iniciando inferência em 4140 amostras...
Inferência concluída.


In [15]:
print("\n--- Relatório de Classificação (Modelo TFLite) ---")
print(classification_report(true_labels, predictions_tflite, target_names=['Classe 0', 'Classe 1', 'Classe 2']))

print("\n--- Matriz de Confusão (Modelo TFLite) ---")
print(confusion_matrix(true_labels, predictions_tflite))


--- Relatório de Classificação (Modelo TFLite) ---
              precision    recall  f1-score   support

    Classe 0       0.66      0.84      0.74      2070
    Classe 1       0.67      0.50      0.57      1029
    Classe 2       0.66      0.46      0.55      1041

    accuracy                           0.66      4140
   macro avg       0.66      0.60      0.62      4140
weighted avg       0.66      0.66      0.65      4140


--- Matriz de Confusão (Modelo TFLite) ---
[[1735  169  166]
 [ 430  517   82]
 [ 468   89  484]]
